In [1]:
import pandas as pd
import sqlite3 as sql
import json
import re
from pathlib import Path
from datetime import datetime,timezone


In [3]:
PROJECT_ROOT = Path.cwd().parent
DB_PATH = PROJECT_ROOT / "data" / "processed" / "chatbot.db"
MAX_FILE_SIZE_MB = 100

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DB_PATH:", DB_PATH)
print("MAX_FILE_SIZE_MB:", MAX_FILE_SIZE_MB)


PROJECT_ROOT: d:\ChatBot test
DB_PATH: d:\ChatBot test\data\processed\chatbot.db
MAX_FILE_SIZE_MB: 100


In [10]:
raw_file=PROJECT_ROOT / "data" / "raw" / "Online Retail.xlsx"
retail_raw= pd.read_excel(raw_file)


In [11]:
retail_raw.shape

(541909, 8)

In [12]:
retail_raw.head()

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
0,536365,85123A,WHITE HANGING HEART T-LIGHT HOLDER,6,2010-12-01 08:26:00,2.55,17850.0,United Kingdom
1,536365,71053,WHITE METAL LANTERN,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
2,536365,84406B,CREAM CUPID HEARTS COAT HANGER,8,2010-12-01 08:26:00,2.75,17850.0,United Kingdom
3,536365,84029G,KNITTED UNION FLAG HOT WATER BOTTLE,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom
4,536365,84029E,RED WOOLLY HOTTIE WHITE HEART.,6,2010-12-01 08:26:00,3.39,17850.0,United Kingdom


In [13]:
retail_raw.dtypes

InvoiceNo              object
StockCode              object
Description            object
Quantity                int64
InvoiceDate    datetime64[us]
UnitPrice             float64
CustomerID            float64
Country                   str
dtype: object

In [14]:
retail_raw.isna().sum()

InvoiceNo           0
StockCode           0
Description      1454
Quantity            0
InvoiceDate         0
UnitPrice           0
CustomerID     135080
Country             0
dtype: int64

In [15]:
for col in ["InvoiceNo", "StockCode"]:
    print(col, "dtype:", retail_raw[col].dtype)
    print(retail_raw[col].map(type).value_counts(), "\n")

InvoiceNo dtype: object
InvoiceNo
<class 'int'>    532618
<class 'str'>      9291
Name: count, dtype: int64 

StockCode dtype: object
StockCode
<class 'int'>    487036
<class 'str'>     54873
Name: count, dtype: int64 



In [16]:
non_numeric = retail_raw[~retail_raw["StockCode"].map(lambda v: isinstance(v, int))]
non_numeric["StockCode"].unique()[:20]

array(['85123A', '84406B', '84029G', '84029E', 'POST', '82494L', '85099C',
       '84997B', '84997C', '84519A', '85183B', '85071B', '37444A',
       '37444C', '84971S', '15056BL', '15056N', 'D', '35004C', '85049A'],
      dtype=object)

In [17]:
is_cancelled = retail_raw["InvoiceNo"].astype(str).str.startswith("C")
print("Cancelled rows:", is_cancelled.sum())
retail_raw[is_cancelled].head()


Cancelled rows: 9288


,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
141,C536379,D,Discount,-1,2010-12-01 09:41:00,27.50,14527.0,United Kingdom
154,C536383,35004C,SET OF 3 COLOURED FLYING DUCKS,-1,2010-12-01 09:49:00,4.65,15311.0,United Kingdom
235,C536391,22556,PLASTERS IN TIN CIRCUS PARADE,-12,2010-12-01 10:24:00,1.65,17548.0,United Kingdom
236,C536391,21984,PACK OF 12 PINK PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom
237,C536391,21983,PACK OF 12 BLUE PAISLEY TISSUES,-24,2010-12-01 10:24:00,0.29,17548.0,United Kingdom


In [18]:
retail_raw.loc[is_cancelled, "Quantity"].describe()


count     9288.000000
mean       -29.885228
std       1145.786965
min     -80995.000000
25%         -6.000000
50%         -2.000000
75%         -1.000000
max         -1.000000
Name: Quantity, dtype: float64

In [19]:
retail_raw[["UnitPrice", "Quantity"]].min()

UnitPrice   -11062.06
Quantity    -80995.00
dtype: float64

In [20]:
retail_raw[retail_raw["UnitPrice"] <= 0].head(10)

,InvoiceNo,StockCode,Description,Quantity,InvoiceDate,UnitPrice,CustomerID,Country
622,536414,22139,NaN,56,2010-12-01 11:52:00,0.0,NaN,United Kingdom
1970,536545,21134,NaN,1,2010-12-01 14:32:00,0.0,NaN,United Kingdom
1971,536546,22145,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1972,536547,37509,NaN,1,2010-12-01 14:33:00,0.0,NaN,United Kingdom
1987,536549,85226A,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
1988,536550,85044,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2024,536552,20950,NaN,1,2010-12-01 14:34:00,0.0,NaN,United Kingdom
2025,536553,37461,NaN,3,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2026,536554,84670,NaN,23,2010-12-01 14:35:00,0.0,NaN,United Kingdom
2406,536589,21777,NaN,-10,2010-12-01 16:50:00,0.0,NaN,United Kingdom
